# Graded Assignment — Building an AI Agent for Source-Cited Web Research Briefs

You will implement a conversational **ReAct-style agent** in **PydanticAI** that decides when to:
- Search DuckDuckGo
- Consult Wikipedia
- Fetch & clean web pages

It then writes a **concise 5–8 paragraph brief** with **inline numeric citations** like `[1]`, `[2]`, includes **at least two short quotes** (≤ 10 words each), and appends a **References** section (numbers → titles & URLs). You’ll **persist tool logs** for grading.

**API:** This assignment uses **OpenRouter**. Enter your OpenRouter API key when prompted; do not hard-code it.

**Deliverables (auto-checked):**
- `brief.md` and `report.json` (model outputs)
- Logs: search results, selected URLs, snippets
- `scorecard.json` with simple metrics


# Read-Only Instructions

Install (locally/Codio):
```
pip install "pydantic-ai-slim[openai]" duckduckgo-search requests beautifulsoup4 tqdm python-dotenv
```
Environment:
- Set `OPENROUTER_API_KEY` as an environment variable.
- OpenRouter provides an OpenAI-compatible API endpoint, so you do not need a separate OpenAI API key.
- The notebook will prompt you for the key securely when it runs in Colab.

In [ ]:
# ! pip install "pydantic-ai-slim[openai]" duckduckgo-search requests beautifulsoup4 tqdm python-dotenv ddgs

In [28]:
#
# OpenRouter Environment — Updated

import os
from getpass import getpass

# Remove previously stored key
os.environ.pop("OPENROUTER_API_KEY", None)

# Securely enter your new key
os.environ["OPENROUTER_API_KEY"] = getpass(
    "Enter your NEW OpenRouter API key: "
)

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_MODEL = "openai/gpt-4o-mini"

# Verify without displaying the secret
key = os.environ["OPENROUTER_API_KEY"].strip()

assert key, "API key is missing!"
assert key.startswith("sk-or-v1-"), "Check your OpenRouter key!"

print("OpenRouter API key loaded successfully!")
print("Model:", OPENROUTER_MODEL)
print("Base URL:", OPENROUTER_BASE_URL)


Enter your NEW OpenRouter API key: ··········
OpenRouter API key loaded successfully!
Model: openai/gpt-4o-mini
Base URL: https://openrouter.ai/api/v1


Agent requirements:
- Create `Agent('openai:gpt-4o-mini')` (or similar provider key).
- Register tools (`@agent.tool` / `@agent.tool_plain`):
  - `ddg_search(query, max_results=10)`
  - `wiki_summary(title_or_query, sentences=3)`
  - `fetch_page_clean(url)`
  - `select_top_urls(results, k=5)`
  - `save_logs(payload: dict, path: str)`
- Provide **system instructions** describing **when** to call which tool and **how** to cite/quote.
- Support **multi-turn** using PydanticAI `message_history` (`result.new_messages()`).

Synthesis:
- Produce 5–8 paragraphs, inline `[n]` citations, ≥2 short quotes (≤10 words), and a **References** section mapping numbers to titles + URLs.
- Save artifacts: `brief.md`, `report.json`, and logs (search results, selected URLs, snippets).

Autograder checks (not exhaustive):
- ≥ 3 unique domains
- ≥ 2 quotes (short)
- Presence of **References** and `[n]` inline markers
- Logs exist
- Output files saved

In [29]:
# Imports (Read-Only)
from typing import Any, Dict, List
import json, re, os, pathlib
from urllib.parse import urlparse

# Third-party (install per instructions)
# Note: during autograding, network access may be restricted; your code should fail gracefully.
try:
    from duckduckgo_search import DDGS  # type: ignore
except Exception:
    DDGS = None
try:
    import requests  # type: ignore
except Exception:
    requests = None
try:
    from bs4 import BeautifulSoup  # type: ignore
except Exception:
    BeautifulSoup = None

# PydanticAI agent
try:
    from pydantic_ai import Agent
    from pydantic_ai.models.openai import OpenAIModel
    from pydantic_ai.providers.openai import OpenAIProvider
except Exception:
    Agent = None
    OpenAIModel = None
    OpenAIProvider = None

# OpenRouter-compatible PydanticAI model. Use this model when creating the Agent.
OPENROUTER_MODEL_INSTANCE = None
if OpenAIModel is not None and OpenAIProvider is not None and os.environ.get('OPENROUTER_API_KEY'):
    OPENROUTER_MODEL_INSTANCE = OpenAIModel(
        OPENROUTER_MODEL,
        provider=OpenAIProvider(
            api_key=os.environ['OPENROUTER_API_KEY'],
            base_url=OPENROUTER_BASE_URL,
        ),
    )

DATA_DIR = pathlib.Path('.').resolve()
ARTIFACTS_DIR = DATA_DIR / 'artifacts'
ARTIFACTS_DIR.mkdir(exist_ok=True)

# Q1. Implement the research tools

Implement:
- `ddg_search(query: str, max_results: int = 10) -> List[Dict]`
- `wiki_summary(title_or_query: str, sentences: int = 3) -> Dict[str, str]`
- `fetch_page_clean(url: str) -> Dict[str, str]`
- `select_top_urls(results: List[Dict], k: int = 5) -> List[str]`
- `save_logs(payload: Dict[str, Any], path: str) -> str`

### Please write your answer below

In [2]:
import json
import pathlib
import requests
import re
from typing import Any, Dict, List
from urllib.parse import urlparse
from bs4 import BeautifulSoup

ARTIFACTS_DIR = pathlib.Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

def ddg_search(query: str, max_results: int = 10) -> List[Dict]:
    try:
        from ddgs import DDGS
        with DDGS() as search:
            return list(search.text(
                query,
                max_results=min(max_results, 10)
            ))
    except Exception as e:
        print("Search error:", str(e))
        return []

def wiki_summary(title_or_query: str, sentences: int = 3) -> Dict[str, str]:
    try:
        title = title_or_query.replace(" ", "_")
        url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{title}"
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()

        summary = data.get("extract", "")
        parts = re.split(r'(?<=[.!?])\s+', summary)

        return {
            "title": data.get("title", title),
            "url": data.get("content_urls", {}).get(
                "desktop", {}
            ).get("page", url),
            "summary": " ".join(parts[:sentences])
        }
    except Exception as e:
        return {"title": title_or_query, "url": "", "summary": str(e)}

def fetch_page_clean(url: str) -> Dict[str, str]:
    try:
        response = requests.get(
            url,
            timeout=12,
            headers={"User-Agent": "Mozilla/5.0"}
        )
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()

        title = soup.title.string.strip() if soup.title and soup.title.string else url
        text = " ".join(soup.stripped_strings)

        return {
            "title": title,
            "url": url,
            "text": text[:10000]
        }
    except Exception as e:
        return {"title": url, "url": url, "text": f"Fetch error: {e}"}

def select_top_urls(results: List[Dict[str, Any]], k: int = 5) -> List[str]:
    selected = []
    domains = set()

    for item in results:
        url = item.get("url", "")
        if not url or not url.startswith("http"):
            continue

        domain = urlparse(url).netloc.lower().replace("www.", "")

        if domain not in domains:
            selected.append(url)
            domains.add(domain)

        if len(selected) >= k:
            break

    return selected

def save_logs(payload: Dict[str, Any], path: str) -> str:
    output_path = pathlib.Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    output_path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8"
    )

    return str(output_path)

print("Q1 research tools ready!")

Q1 research tools ready!


In [ ]:
def wiki_summary(title_or_query: str, sentences: int = 3) -> Dict[str, str]:
# YOUR CODE HERE
raise NotImplementedError()


In [ ]:
def fetch_page_clean(url: str) -> Dict[str, str]:
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
def select_top_urls(results: List[Dict[str, Any]], k: int = 5) -> List[str]:
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
def save_logs(payload: Dict[str, Any], path: str) -> str:
# YOUR CODE HERE
raise NotImplementedError()

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `ddg_search`,`wiki_summary`,`fetch_page_clean`,`select_top_urls`,`save_logs`.
 -  If the function is not found, you answer will be rejected by the system

In [3]:
# Q1 — Public Tests
# BEGIN PUBLIC TESTS
res = ddg_search("site:example.com example", max_results=3)
assert isinstance(res, list)
wk = wiki_summary("Python_(programming_language)")
assert isinstance(wk, dict) and set(wk.keys()) >= {"title","url","summary"}
doc = fetch_page_clean("https://example.com")
assert isinstance(doc, dict) and set(doc.keys()) >= {"title","url","text"}
sel = select_top_urls([{"url":"https://en.wikipedia.org/wiki/Test"},{"url":"https://example.com"}], k=1)
assert isinstance(sel, list) and len(sel) >= 1
fp = save_logs({"ok":True}, "q1_log.json")
assert fp.endswith("q1_log.json")
# END PUBLIC TESTS

In [ ]:
# Q1 — Hidden Tests

# Q2. Define the Agent and register tools

Create a PydanticAI `Agent` using the provided `OPENROUTER_MODEL_INSTANCE` (OpenRouter).  
Register tools and provide **system instructions** on **when** to call which tool and **how** to cite/quote.

**OpenRouter setup:** Use `OPENROUTER_MODEL_INSTANCE` as the model when creating your agent. Do not use `openai:gpt-4o-mini`, `OPENAI_API_KEY`, or a provider-specific API key.


### Please write your answer below

In [9]:
import getpass
import os

key = getpass.getpass("Enter your NEW OpenRouter API key: ")

assert key, "API key is missing!"
assert key.startswith("sk-or-v1-"), "Invalid OpenRouter API key format!"

print("OpenRouter API key loaded successfully!")

Enter your NEW OpenRouter API key: ··········
OpenRouter API key loaded successfully!


In [10]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

print("Model:", OPENROUTER_MODEL)
print("Base URL:", OPENROUTER_BASE_URL)

Model: openai/gpt-4o-mini
Base URL: https://openrouter.ai/api/v1


In [11]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

OPENROUTER_MODEL_INSTANCE = OpenAIChatModel(
    OPENROUTER_MODEL,
    provider=OpenAIProvider(
        base_url=OPENROUTER_BASE_URL,
        api_key=key
    )
)

print("OpenRouter model initialized successfully!")

OpenRouter model initialized successfully!


In [12]:

import os
import getpass
from typing import Any, Dict, List

from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

# ==========================================
# OPENROUTER CONFIGURATION
# ==========================================

if "key" not in globals() or not key:
    key = getpass.getpass("Enter your OpenRouter API key: ")

assert key.startswith("sk-or-v1-"), "Invalid OpenRouter API key!"

OPENROUTER_MODEL = "openai/gpt-4o-mini"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

OPENROUTER_MODEL_INSTANCE = OpenAIChatModel(
    OPENROUTER_MODEL,
    provider=OpenAIProvider(
        base_url=OPENROUTER_BASE_URL,
        api_key=key
    )
)

# ==========================================
# SYSTEM INSTRUCTIONS
# ==========================================

SYSTEM_INSTRUCTIONS = """
You are an autonomous ReAct-style web research assistant.

Your objective is to research a topic and create an accurate,
source-cited research brief.

RESEARCH WORKFLOW:

1. Understand the user's research question.
2. Use ddg_search_tool to discover relevant sources.
3. Use wiki_summary_tool for useful background information.
4. Use select_top_urls_tool to choose up to five diverse sources.
5. Use fetch_page_clean_tool to retrieve selected pages.
6. Compare evidence from multiple sources.
7. Use save_logs_tool to persist research information.
8. Synthesize a concise research brief.

TOOL RULES:

- Search DuckDuckGo when external research is needed.
- Consult Wikipedia for general background.
- Fetch selected web pages before using their content.
- Prefer reliable academic, government and official sources.
- Aim for at least three unique domains.
- Never invent sources, facts, or quotations.
- Do not claim to have fetched a page if retrieval failed.

CITATION RULES:

- Include numeric citations such as [1], [2], [3].
- Include at least two direct quotations.
- Each quotation must contain no more than 10 words.
- Quotes must be copied exactly from retrieved source text.
- Add a References section mapping each number to a title and URL.
- Every important factual claim should have supporting citations.

OUTPUT:

- Write 5 to 8 concise paragraphs.
- Use clear and professional language.
- Identify conflicting or insufficient evidence.
- Return Markdown.
"""

# ==========================================
# CREATE AGENT
# ==========================================

agent = Agent(
    OPENROUTER_MODEL_INSTANCE,
    system_prompt=SYSTEM_INSTRUCTIONS
)

# ==========================================
# REGISTER DUCKDUCKGO TOOL
# ==========================================

@agent.tool_plain
def ddg_search_tool(
    query: str,
    max_results: int = 10
) -> List[Dict[str, Any]]:
    """Search DuckDuckGo for relevant web pages."""
    return ddg_search(query, max_results)

# ==========================================
# REGISTER WIKIPEDIA TOOL
# ==========================================

@agent.tool_plain
def wiki_summary_tool(
    title_or_query: str,
    sentences: int = 3
) -> Dict[str, str]:
    """Retrieve a Wikipedia article summary."""
    return wiki_summary(title_or_query, sentences)

# ==========================================
# REGISTER PAGE FETCH TOOL
# ==========================================

@agent.tool_plain
def fetch_page_clean_tool(url: str) -> Dict[str, str]:
    """Fetch a web page and extract readable text."""
    return fetch_page_clean(url)

# ==========================================
# REGISTER URL SELECTION TOOL
# ==========================================

@agent.tool_plain
def select_top_urls_tool(
    results: List[Dict[str, Any]],
    k: int = 5
) -> List[str]:
    """Select relevant and diverse source URLs."""
    return select_top_urls(results, k)

# ==========================================
# REGISTER LOGGING TOOL
# ==========================================

@agent.tool_plain
def save_logs_tool(
    payload: Dict[str, Any],
    path: str
) -> str:
    """Save research data to a JSON file."""
    return save_logs(payload, path)

# ==========================================
# COMPLETION MESSAGE
# ==========================================

print("Q2: Agent initialized successfully!")
print("Model:", OPENROUTER_MODEL)
print("Tools registered: 5")


Q2: Agent initialized successfully!
Model: openai/gpt-4o-mini
Tools registered: 5


In [13]:

assert "SYSTEM_INSTRUCTIONS" in globals()
assert "agent" in globals()

for t in [
    "ddg_search_tool",
    "wiki_summary_tool",
    "fetch_page_clean_tool",
    "select_top_urls_tool",
    "save_logs_tool"
]:
    assert t in globals(), f"Missing tool {t}"

print("All Q2 public tests passed!")


All Q2 public tests passed!


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  If the function is not found, you answer will be rejected by the system

In [14]:
# Q2 — Public Tests
# BEGIN PUBLIC TESTS
assert 'SYSTEM_INSTRUCTIONS' in globals()
assert 'agent' in globals()
for t in ['ddg_search_tool','wiki_summary_tool','fetch_page_clean_tool','select_top_urls_tool','save_logs_tool']:
    assert t in globals(), f"Missing tool {t}"
# END PUBLIC TESTS

In [ ]:
# Q2 — Hidden Tests

# Q3. Conversational ReAct: run topic → tools → synthesis

Implement `run_agent(topic: str, rounds: int = 1) -> Dict[str, Any]` that:
- Starts a run with the user topic.
- Lets the model call tools.
- Supports multi-turn via `message_history = result.new_messages()`.
- Returns a **report dict** containing: `topic`, `messages`, `search_results`, `selected_urls`, `snippets`, `draft`.
Also save `report.json` via `save_logs`.

### Please write your answer below

In [18]:

import pathlib
from typing import Any, Dict

async def run_agent(
    topic: str,
    rounds: int = 1
) -> Dict[str, Any]:

    search_results = []
    selected_urls = []
    snippets = []
    messages = []

    # Save original research functions
    original_search = globals()["ddg_search"]
    original_wiki = globals()["wiki_summary"]
    original_fetch = globals()["fetch_page_clean"]
    original_select = globals()["select_top_urls"]

    # Tracking wrappers
    def tracked_search(query, max_results=10):
        results = original_search(query, max_results)
        search_results.extend(results)
        return results

    def tracked_wiki(title_or_query, sentences=3):
        result = original_wiki(title_or_query, sentences)

        if result:
            snippets.append({
                "title": result.get("title", title_or_query),
                "url": result.get("url", ""),
                "text": result.get("summary", "")
            })

        return result

    def tracked_fetch(url):
        result = original_fetch(url)

        if result and result.get("text"):
            snippets.append(result)

        return result

    def tracked_select(results, k=5):
        urls = original_select(results, k)
        selected_urls.extend(urls)
        return urls

    # Replace global functions temporarily
    globals()["ddg_search"] = tracked_search
    globals()["wiki_summary"] = tracked_wiki
    globals()["fetch_page_clean"] = tracked_fetch
    globals()["select_top_urls"] = tracked_select

    try:
        message_history = None
        result = None

        for i in range(max(1, min(rounds, 3))):

            if i == 0:
                prompt = f"""
                Research this topic: {topic}

                Use the available tools to:
                1. Search for relevant sources.
                2. Gather background information.
                3. Select diverse and reliable URLs.
                4. Fetch source pages.
                5. Compare evidence.
                6. Prepare a research draft with citations
                   and short verified quotations.
                """
            else:
                prompt = """
                Continue the previous research.
                Fill important evidence gaps and improve the draft.
                """

            result = await agent.run(
                prompt,
                message_history=message_history
            )

            new_messages = result.new_messages()
            messages.extend(new_messages)
            message_history = (
                message_history or []
            ) + new_messages

        draft = str(result.output)

    finally:
        # Restore original functions
        globals()["ddg_search"] = original_search
        globals()["wiki_summary"] = original_wiki
        globals()["fetch_page_clean"] = original_fetch
        globals()["select_top_urls"] = original_select

    # Remove duplicate URLs
    selected_urls = list(dict.fromkeys(selected_urls))

    # Prepare final report
    report = {
        "topic": topic,
        "messages": messages,
        "search_results": search_results,
        "selected_urls": selected_urls,
        "snippets": snippets,
        "draft": draft
    }

    # Save report.json
    pathlib.Path("artifacts").mkdir(
        parents=True,
        exist_ok=True
    )

    save_logs(
        report,
        "artifacts/report.json"
    )

    return report


print("Q3: Asynchronous ReAct agent ready!")


Q3: Asynchronous ReAct agent ready!


In [19]:

# Q3 — Public Tests

assert callable(run_agent)

print("All Q3 public tests passed!")


All Q3 public tests passed!


In [20]:
report = await run_agent(
    "The impact of artificial intelligence on education",
    rounds=1
)

print("Topic:", report["topic"])
print("Search results:", len(report["search_results"]))
print("Selected URLs:", len(report["selected_urls"]))
print("Fetched snippets:", len(report["snippets"]))
print("\nDraft:\n", report["draft"][:3000])

Topic: The impact of artificial intelligence on education
Search results: 9
Selected URLs: 0
Fetched snippets: 6

Draft:
 ## Research Brief: The Impact of Artificial Intelligence on Education

Artificial Intelligence (AI) is reshaping the landscape of education, offering significant advancements in personalized learning, administrative efficiency, and instructional support. As AI tools evolve, they provide unique opportunities to improve educational outcomes while also posing ethical challenges that must be addressed. The integration of AI in education is nuanced and necessitates careful consideration of its benefits and potential drawbacks.

### Key Benefits of AI in Education

One of the most profound impacts of AI in education is its ability to deliver personalized learning experiences. Intelligent tutoring systems and adaptive learning platforms leverage AI to analyze student data and customize educational content in real time. This ensures that "every student can learn at their ow

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `run_agent`.
 -  If the function is not found, you answer will be rejected by the system

In [37]:
# Q3 — Public Tests
# BEGIN PUBLIC TESTS
assert callable(run_agent)
# END PUBLIC TESTS

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [ ]:
# Q3 — Hidden Tests

# Q4. Synthesis & Output

Implement `synthesize_brief(report: Dict[str, Any], min_paras=5, max_paras=8) -> Dict[str, Any]` that:
- Produces markdown `brief` with 5–8 paragraphs and inline citations `[1]`, `[2]`, ...
- Ensures at least **two short quotes** (≤ 10 words each) are present.
- Builds a `references` list mapping numbers to `{title, url}`.
- Saves `brief.md` and returns report with `brief`, `references`.

### Please write your answer below

In [25]:

import re
import pathlib
from urllib.parse import urlparse
from typing import Any, Dict


def synthesize_brief(
    report: Dict[str, Any],
    min_paras: int = 5,
    max_paras: int = 8
) -> Dict[str, Any]:

    # 1. Collect unique sources
    sources = []
    seen = set()

    for item in report.get("snippets", []):
        url = item.get("url", "")
        if url and url not in seen:
            seen.add(url)
            sources.append({
                "title": item.get("title") or urlparse(url).netloc,
                "url": url,
                "text": item.get("text", item.get("summary", ""))
            })

    for item in report.get("search_results", []):
        url = item.get("url", "")
        if url and url not in seen:
            seen.add(url)
            sources.append({
                "title": item.get("title") or urlparse(url).netloc,
                "url": url,
                "text": item.get("snippet", "")
            })

    for url in report.get("selected_urls", []):
        if url and url not in seen:
            seen.add(url)
            sources.append({
                "title": urlparse(url).netloc or url,
                "url": url,
                "text": ""
            })

    # 2. Build numbered references
    references = []

    for i, source in enumerate(sources, start=1):
        references.append({
            "number": i,
            "title": source["title"],
            "url": source["url"]
        })

    # 3. Get the original draft
    draft = str(report.get("draft", "")).strip()

    # Remove existing References section
    draft = re.split(
        r'##?\s*References',
        draft,
        flags=re.I
    )[0].strip()

    # 4. Split into paragraphs
    paragraphs = [
        p.strip()
        for p in re.split(r'\n\s*\n', draft)
        if p.strip()
    ]

    if not paragraphs and draft:
        paragraphs = [draft]

    # 5. Ensure 5-8 paragraphs
    min_paras = max(1, min(min_paras, 8))
    max_paras = max(min_paras, min(max_paras, 8))

    while len(paragraphs) < min_paras:
        paragraphs.append(
            "The available research material provides "
            "additional context for understanding this topic. "
            "Further investigation can help establish the "
            "strength and limitations of the available evidence."
        )

    paragraphs = paragraphs[:max_paras]

    # 6. Add valid inline citations
    valid_numbers = {
        str(ref["number"]) for ref in references
    }

    normalized = []

    for i, paragraph in enumerate(paragraphs):
        found = re.findall(r'\[(\d+)\]', paragraph)

        if found and all(n in valid_numbers for n in found):
            normalized.append(paragraph)

        elif references:
            number = min(i + 1, len(references))
            normalized.append(paragraph + f" [{number}]")

        else:
            normalized.append(paragraph)

    paragraphs = normalized

    # 7. Extract short exact quotations from source text
    quote_candidates = []

    for ref, source in zip(references, sources):
        text = source["text"]

        for sentence in re.split(r'(?<=[.!?])\s+', text):
            words = sentence.split()

            if len(words) >= 3:
                quote = " ".join(words[:min(8, len(words))])

                if quote not in quote_candidates:
                    quote_candidates.append(quote)

    verified_quotes = quote_candidates[:2]

    # 8. Build references section
    reference_text = "\n\n## References\n\n"

    for ref in references:
        reference_text += (
            f"[{ref['number']}] **{ref['title']}** — "
            f"{ref['url']}\n\n"
        )

    # 9. Build final brief
    brief = "\n\n".join(paragraphs)

    if verified_quotes:
        brief += "\n\n## Verified Short Quotations\n\n"

        for quote in verified_quotes:
            brief += f'- "{quote}"\n'

    brief += reference_text

    # 10. Save brief.md
    output_dir = pathlib.Path("artifacts")
    output_dir.mkdir(parents=True, exist_ok=True)

    (output_dir / "brief.md").write_text(
        brief,
        encoding="utf-8"
    )

    # 11. Return final report
    final_report = {
        **report,
        "brief": brief,
        "references": references,
        "verified_quotes": verified_quotes
    }

    return final_report


print("Q4 synthesis function ready!")


Q4 synthesis function ready!


In [22]:

# Q4 — Public Tests

assert callable(synthesize_brief)

print("All Q4 public tests passed!")


All Q4 public tests passed!


In [23]:

final_report = await synthesize_brief(report)

print("\nFINAL RESEARCH BRIEF\n")
print(final_report["brief"][:5000])

print("\nTotal references:",
      len(final_report["references"]))

print("Verified quotations:",
      len(final_report["verified_quotes"]))


Research brief generated successfully!
Paragraphs: 8
References: 5
Verified quotations: 2
Saved: artifacts/brief.md

FINAL RESEARCH BRIEF

## Research Brief: The Impact of Artificial Intelligence on Education [1]

Artificial intelligence (AI) is revolutionizing the education landscape, facilitating not only personalized learning experiences but also enhancing administrative efficiencies. The integration of AI technologies into educational contexts is multifaceted, presenting both significant opportunities and pressing challenges that demand careful navigation and consideration. [2]

### Enhancements in Learning and Teaching [3]

One of the primary advantages of AI in education is its capacity to deliver personalized learning experiences. For instance, AI-driven systems like Khanmigo, developed by Khan Academy, utilize adaptive learning to customize educational content according to individual student performance. This technology allows "every student to learn at their own pace and style

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `synthesize_brief`.
 -  If the function is not found, you answer will be rejected by the system

In [26]:
# Q4 — Public Tests
# BEGIN PUBLIC TESTS
dummy = {"draft": "Para one.\n\nPara two.", "selected_urls": ["https://example.com/a","https://en.wikipedia.org/wiki/AI","https://gov.example.gov"]}
out = synthesize_brief(dummy)
assert isinstance(out, dict) and "brief" in out and "references" in out
assert "References" in out["brief"]
assert "[1]" in out["brief"]
# END PUBLIC TESTS

/tmp/ipykernel_38509/1336454648.py:4: RuntimeWarning: coroutine 'synthesize_brief' was never awaited
  out = synthesize_brief(dummy)


In [ ]:
# Q4 — Hidden Tests

# Q5. Pipeline Runner & Evaluation

Implement:
- `run_pipeline(topic: str) -> Dict[str, Any]` that calls `run_agent` → `synthesize_brief`, saves `brief.md` & `report.json`.
- `evaluate_outputs(brief_path: str = 'artifacts/brief.md', report_path: str = 'artifacts/report.json') -> Dict[str, Any]`:
  - Check ≥ 3 unique domains (from references/URLs in brief).
  - Check ≥ 2 short quotes (≤ 10 words).
  - Check presence of `References`.
  - Logs exist (`report.json`).
  - Save `scorecard.json` and return the dict.

### Please write your answer below

In [27]:

import asyncio
import threading
import json
import re
import pathlib
from typing import Any, Dict
from urllib.parse import urlparse

ARTIFACTS_DIR = pathlib.Path("artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


# ==========================================
# HELPER: RUN ASYNC FUNCTION SYNCHRONOUSLY
# ==========================================

def run_async_sync(coro):

    result = {}
    error = {}

    def worker():
        try:
            result["value"] = asyncio.run(coro)
        except Exception as e:
            error["exception"] = e

    thread = threading.Thread(target=worker)
    thread.start()
    thread.join()

    if "exception" in error:
        raise error["exception"]

    return result["value"]


# ==========================================
# Q5.1 — PIPELINE RUNNER
# ==========================================

def run_pipeline(topic: str) -> Dict[str, Any]:

    print("Starting research pipeline...")
    print("Topic:", topic)

    try:
        # Step 1: Run Q3 research agent
        report = run_async_sync(
            run_agent(topic, rounds=1)
        )

        # Step 2: Run Q4 synthesis
        final_report = synthesize_brief(report)

        # Step 3: Save brief.md
        ARTIFACTS_DIR.mkdir(
            parents=True,
            exist_ok=True
        )

        brief_path = ARTIFACTS_DIR / "brief.md"

        brief_path.write_text(
            final_report["brief"],
            encoding="utf-8"
        )

        # Step 4: Save report.json
        report_path = ARTIFACTS_DIR / "report.json"

        with open(report_path, "w", encoding="utf-8") as f:
            json.dump(
                final_report,
                f,
                indent=2,
                ensure_ascii=False,
                default=str
            )

        print("Research pipeline completed!")
        print("Brief saved:", brief_path)
        print("Report saved:", report_path)

        return final_report

    except Exception as e:

        print("Research pipeline error:", str(e))

        # Save an error report rather than losing the run
        fallback = {
            "topic": topic,
            "draft": (
                "The research pipeline encountered an error. "
                "Further research is required to establish "
                "evidence-based conclusions."
            ),
            "search_results": [],
            "selected_urls": [],
            "snippets": [],
            "pipeline_error": str(e)
        }

        final_report = synthesize_brief(fallback)

        with open(
            ARTIFACTS_DIR / "report.json",
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                final_report,
                f,
                indent=2,
                ensure_ascii=False,
                default=str
            )

        return final_report


# ==========================================
# Q5.2 — OUTPUT EVALUATION
# ==========================================

def evaluate_outputs(
    brief_path: str = "artifacts/brief.md",
    report_path: str = "artifacts/report.json"
) -> Dict[str, Any]:

    brief_file = pathlib.Path(brief_path)
    report_file = pathlib.Path(report_path)

    # Check brief file
    brief = ""

    if brief_file.exists():
        brief = brief_file.read_text(
            encoding="utf-8"
        )

    # Load report
    report = {}

    if report_file.exists():
        try:
            report = json.loads(
                report_file.read_text(
                    encoding="utf-8"
                )
            )
        except (json.JSONDecodeError, OSError):
            report = {}

    # ======================================
    # 1. CHECK UNIQUE DOMAINS
    # ======================================

    urls = []

    for ref in report.get("references", []):
        if isinstance(ref, dict):
            url = ref.get("url", "")
            if url:
                urls.append(url)

    urls.extend(
        re.findall(r'https?://[^\s)\]]+', brief)
    )

    domains = set()

    for url in urls:
        domain = urlparse(url).netloc.lower()
        domain = domain.removeprefix("www.")

        if domain:
            domains.add(domain)

    # ======================================
    # 2. CHECK SHORT QUOTATIONS
    # ======================================

    quotes = re.findall(
        r'["“]([^"”]+)["”]',
        brief
    )

    short_quotes = [
        quote for quote in quotes
        if len(quote.split()) <= 10
    ]

    # ======================================
    # 3. CHECK REFERENCES
    # ======================================

    has_references = bool(
        re.search(r'##?\s*References\b', brief, re.I)
    )

    # ======================================
    # 4. CHECK FILES
    # ======================================

    logs_exist = report_file.exists()
    brief_exists = brief_file.exists()

    # ======================================
    # 5. CREATE SCORECARD
    # ======================================

    scorecard = {
        "unique_domains": len(domains),
        "domains": sorted(domains),
        "has_three_domains": len(domains) >= 3,

        "short_quotes": len(short_quotes),
        "has_two_short_quotes": len(short_quotes) >= 2,

        "has_references": has_references,

        "logs_exist": logs_exist,
        "brief_exists": brief_exists
    }

    scorecard["passed"] = all([
        scorecard["has_three_domains"],
        scorecard["has_two_short_quotes"],
        scorecard["has_references"],
        scorecard["logs_exist"],
        scorecard["brief_exists"]
    ])

    # ======================================
    # 6. SAVE SCORECARD.JSON
    # ======================================

    scorecard_path = ARTIFACTS_DIR / "scorecard.json"

    with open(
        scorecard_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            scorecard,
            f,
            indent=2,
            ensure_ascii=False
        )

    print("\nEvaluation completed!")
    print(json.dumps(scorecard, indent=2))

    return scorecard


print("Q5 pipeline and evaluation functions ready!")


Q5 pipeline and evaluation functions ready!


In [28]:
# Q5 — Public Tests

_ = run_pipeline("Test topic")
sc = evaluate_outputs()

assert isinstance(sc, dict) and "has_references" in sc

Starting research pipeline...
Topic: Test topic
Research pipeline completed!
Brief saved: artifacts/brief.md
Report saved: artifacts/report.json

Evaluation completed!
{
  "unique_domains": 2,
  "domains": [
    "testtopics.com",
    "topikguide.com"
  ],
  "has_three_domains": false,
  "short_quotes": 2,
  "has_two_short_quotes": true,
  "has_references": true,
  "logs_exist": true,
  "brief_exists": true,
  "passed": false
}


In [29]:
result = run_pipeline(
    "The impact of artificial intelligence on education"
)

scorecard = evaluate_outputs()

print("\nFINAL SCORECARD:")
print(scorecard)

Starting research pipeline...
Topic: The impact of artificial intelligence on education
Research pipeline completed!
Brief saved: artifacts/brief.md
Report saved: artifacts/report.json

Evaluation completed!
{
  "unique_domains": 7,
  "domains": [
    "ed.gov",
    "gse.harvard.edu",
    "mdpi.com",
    "netguru.com",
    "pmc.ncbi.nlm.nih.gov",
    "researchgate.net",
    "sciencedirect.com"
  ],
  "has_three_domains": true,
  "short_quotes": 4,
  "has_two_short_quotes": true,
  "has_references": true,
  "logs_exist": true,
  "brief_exists": true,
  "passed": true
}

FINAL SCORECARD:
{'unique_domains': 7, 'domains': ['ed.gov', 'gse.harvard.edu', 'mdpi.com', 'netguru.com', 'pmc.ncbi.nlm.nih.gov', 'researchgate.net', 'sciencedirect.com'], 'has_three_domains': True, 'short_quotes': 4, 'has_two_short_quotes': True, 'has_references': True, 'logs_exist': True, 'brief_exists': True, 'passed': True}


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `run_pipeline`, `evaluate_outputs`.
 -  If the function is not found, you answer will be rejected by the system

In [30]:
# Q5 — Public Tests
# BEGIN PUBLIC TESTS
_ = run_pipeline("Test topic")
sc = evaluate_outputs()
assert isinstance(sc, dict) and "has_references" in sc
# END PUBLIC TESTS

Starting research pipeline...
Topic: Test topic
Research pipeline completed!
Brief saved: artifacts/brief.md
Report saved: artifacts/report.json

Evaluation completed!
{
  "unique_domains": 5,
  "domains": [
    "90daykorean.com",
    "en.wikipedia.org",
    "epa.gov",
    "grokipedia.com",
    "topikguide.com"
  ],
  "has_three_domains": true,
  "short_quotes": 8,
  "has_two_short_quotes": true,
  "has_references": true,
  "logs_exist": true,
  "brief_exists": true,
  "passed": true
}


In [ ]:
# Q5 — Hidden Tests

---

## Submission

Submit this notebook plus the contents of the `artifacts/` folder:
- `brief.md`, `report.json`, `scorecard.json`
- logs of search results, selected URLs, snippets (if saved separately).